<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/append_to_Duckdb_dbase%20from_%20pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
import pandas as pd
import numpy as np

source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):

    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False

        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")


extract_all_zips(source_folder, levelone_folder)

In [4]:
import pandas as pd
import glob
import os

# Path to folder containing CSV files
folder_path = "/content/levelone"

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and append
df_list = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

# Combine all into one dataframe
full_data = pd.concat(df_list, ignore_index=True)



In [6]:
full_data.shape

(1101375, 34)

In [7]:
# Keep only Cash Market segment
full_data = full_data[full_data["Sgmt"] == "CM"]

# Keep only Equity series
full_data = full_data[full_data["SctySrs"] == "EQ"]
'''
# Select required columns
full_data = full_data[[
    "TckrSymb",
    "TradDt",
    "OpnPric",
    "HghPric",
    "LwPric",
    "ClsPric",
    "TtlTradgVol"
]]

# Rename to clean structure
full_data.columns = [
    "symbol",
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume"
]
# Fix date

full_data["date"] = pd.to_datetime(full_data["date"])

print("Equity data cleaned.")
print(full_data.head())
'''

'\n# Select required columns\nfull_data = full_data[[\n    "TckrSymb",\n    "TradDt",\n    "OpnPric",\n    "HghPric",\n    "LwPric",\n    "ClsPric",\n    "TtlTradgVol"\n]]\n\n# Rename to clean structure\nfull_data.columns = [\n    "symbol",\n    "date",\n    "open",\n    "high",\n    "low",\n    "close",\n    "volume"\n]\n# Fix date\n\nfull_data["date"] = pd.to_datetime(full_data["date"])\n\nprint("Equity data cleaned.")\nprint(full_data.head())\n'

In [8]:
full_data.shape

(756692, 34)

In [9]:
import duckdb

# -----------------------------
# CONFIG
# -----------------------------
db_path = "/content/nifty_200_data.duckdb"
table_name = "equities"   # Change if your table name is different

# -----------------------------
# 1. Connect to DuckDB
# -----------------------------
con = duckdb.connect(db_path)

# -----------------------------
# 2. Create Table If Not Exists
# -----------------------------
con.execute(f"""
CREATE TABLE IF NOT EXISTS {table_name} AS
SELECT * FROM full_data WHERE 1=0
""")

# -----------------------------
# 3. Register pandas dataframe
# -----------------------------
con.register("temp_df", full_data)

# -----------------------------
# 4. Append Only New Rows
# -----------------------------
con.execute(f"""
INSERT INTO {table_name}
SELECT * FROM temp_df
WHERE (TckrSymb, TradDt) NOT IN (
    SELECT TckrSymb, TradDt FROM {table_name}
)
""")

print("Append complete.")

Append complete.


In [10]:
# Connect to your database file
con = duckdb.connect(db_path)

# Read equities table into pandas
df = con.execute("SELECT * FROM equities").fetchdf()

con.close()

print(df.head())
print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       TradDt       BizDt Sgmt  Src FinInstrmTp  FinInstrmId          ISIN  \
0  2025-06-06  2025-06-06   CM  NSE         STK        16921  INE144J01027   
1  2025-06-06  2025-06-06   CM  NSE         STK            4  INE253B01015   
2  2025-06-06  2025-06-06   CM  NSE         STK        13061  INE466L01038   
3  2025-06-06  2025-06-06   CM  NSE         STK        30105  INF579M01BB5   
4  2025-06-06  2025-06-06   CM  NSE         STK       756150  INF579M01BC3   

     TckrSymb SctySrs  XpryDt  ...  TtlTradgVol     TtlTrfVal  \
0   20MICRONS      EQ     NaN  ...       167624  3.822951e+07   
1  21STCENMGM      EQ     NaN  ...        11278  7.509231e+05   
2      360ONE      EQ     NaN  ...      2406203  2.620693e+09   
3     GOLD360      EQ     NaN  ...         3454  3.336880e+05   
4   SILVER360      EQ     NaN  ...         2720  2.859651e+05   

   TtlNbOfTxsExctd SsnId  NewBrdLotQty  Rmks  Rsvd1  Rsvd2  Rsvd3  Rsvd4  
0             5008    F1             1   NaN    NaN    NaN    NaN

In [14]:
df.tail(5).to_csv('abc.csv')

In [15]:
df.head(5).to_csv('xyz.csv')